# Topic 5 - Python Functions, Dicts, Lists, and JSON

This notebook is a hands-on practice session for working independently with structured data.

The central workflow is:

```text
JSON file -> Python data -> transformation function -> tests -> JSON file
```

Try each exercise before looking at the solution cell. For this topic, write the implementation yourself without autocomplete or Copilot.

## Learning objectives

By the end, you should be able to:

- define functions with parameters and return values
- read and transform lists, dictionaries, and nested data
- load JSON text from a file and dump Python data back to JSON
- write plain `assert` tests for transformation behavior
- explain the difference between JSON text and a Python object

## 1. Functions return values

A function packages logic behind a name. `return` gives a value back to the caller; `print` only displays a value.

In [ ]:
def add(first_number, second_number):
    return first_number + second_number

def square(number):
    return number * number

assert add(2, 3) == 5
assert square(4) == 16
print(add(2, 3))
print(square(4))

## 2. Lists and dictionaries

Structured data is easiest to manipulate when you first identify its shape. This value is a list containing dictionaries.

In [ ]:
users = [
    {'id': 1, 'name': 'Alice', 'active': True},
    {'id': 2, 'name': 'Bob', 'active': False},
    {'id': 3, 'name': 'Chen', 'active': True},
]

print(type(users).__name__)
print(type(users[0]).__name__)
print(users[0]['name'])

active_names = []
for user in users:
    if user['active']:
        active_names.append(user['name'])

assert active_names == ['Alice', 'Chen']
active_names

## 3. Write a transformation function

The function below keeps active users and returns only the fields needed by another part of an application. The original input is not modified.

In [ ]:
def active_user_summaries(records):
    summaries = []
    for record in records:
        if record['active']:
            summaries.append({
                'id': record['id'],
                'name': record['name'],
            })
    return summaries

summaries = active_user_summaries(users)
summaries

### Practice: inspect the data shape

Before changing a nested value, answer these questions in your own words:

1. What is the outer type of `users`?
2. What is the type of `users[0]`?
3. What type is stored under the `active` key?
4. What would happen if the input list were empty?

In [ ]:
assert active_user_summaries([]) == []
assert active_user_summaries(users) == [
    {'id': 1, 'name': 'Alice'},
    {'id': 3, 'name': 'Chen'},
]

## 4. JSON text versus Python data

JSON is text used for storage and exchange. After parsing, Python has a list or dictionary in memory. `json.loads` parses a string; `json.dumps` serializes Python data into a string.

In [ ]:
import json

json_text = '[{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]'
records_from_text = json.loads(json_text)

assert isinstance(json_text, str)
assert isinstance(records_from_text, list)
assert records_from_text[0]['name'] == 'Alice'

serialized_text = json.dumps(records_from_text, indent=2)
print(serialized_text)

## 5. Separate file handling from transformation

The transformation function accepts Python data. A separate function handles the JSON file boundary. This makes the transformation easy to test without needing a file every time.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

def transform_json_file(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as input_file:
        records = json.load(input_file)

    transformed_records = active_user_summaries(records)

    with open(output_path, 'w', encoding='utf-8') as output_file:
        json.dump(transformed_records, output_file, indent=2)

    return transformed_records

with TemporaryDirectory() as temporary_directory:
    input_path = Path(temporary_directory) / 'users.json'
    output_path = Path(temporary_directory) / 'active_users.json'

    with open(input_path, 'w', encoding='utf-8') as input_file:
        json.dump(users, input_file)

    result = transform_json_file(input_path, output_path)

    with open(output_path, 'r', encoding='utf-8') as output_file:
        saved_result = json.load(output_file)

    assert result == saved_result
    assert saved_result == summaries
    print(output_path.read_text(encoding='utf-8'))

## 6. Tests prove correctness

A printed result can look right without checking all requirements. These tests use plain `assert` statements to make expected behavior explicit. The function names also follow pytest's discovery convention.

In [ ]:
def test_active_user_summaries_filters_inactive_records():
    input_records = [
        {'id': 10, 'name': 'Ada', 'active': True},
        {'id': 11, 'name': 'Grace', 'active': False},
    ]
    assert active_user_summaries(input_records) == [
        {'id': 10, 'name': 'Ada'},
    ]

def test_active_user_summaries_handles_empty_input():
    assert active_user_summaries([]) == []

def test_active_user_summaries_does_not_modify_input():
    input_records = [{'id': 1, 'name': 'A', 'active': True}]
    original_records = input_records.copy()
    active_user_summaries(input_records)
    assert input_records == original_records

test_active_user_summaries_filters_inactive_records()
test_active_user_summaries_handles_empty_input()
test_active_user_summaries_does_not_modify_input()
print('All notebook tests passed.')

### Running the same tests with pytest

In a project, put the function in a Python module and these `test_...` functions in a file such as `test_transformations.py`. Then run:

```text
python -m pytest
```

A virtual environment keeps pytest and other project dependencies isolated:

```text
python -m venv .venv
.venv\Scripts\activate
python -m pip install pytest
```

## 7. Final challenge

Write this function without autocomplete:

```python
def index_by_id(records):
    ...
```

Requirements:

- accept a list of dictionaries
- use each record's `id` as a key
- store the full record as the value
- return an empty dictionary for an empty list
- write at least two `assert` tests before considering it complete

In [ ]:
def index_by_id(records):
    indexed_records = {}
    for record in records:
        indexed_records[record['id']] = record
    return indexed_records

assert index_by_id([]) == {}
assert index_by_id(users)[2]['name'] == 'Bob'
assert index_by_id(users) == {
    1: {'id': 1, 'name': 'Alice', 'active': True},
    2: {'id': 2, 'name': 'Bob', 'active': False},
    3: {'id': 3, 'name': 'Chen', 'active': True},
}
print('Final challenge passed.')

## Completion check

Before marking Topic 5 complete, confirm that you can independently:

- identify the shape of unfamiliar JSON data
- write a function that transforms it
- test expected and edge-case behavior with `assert`
- read a failing assertion and compare expected with actual
- load and save the transformed data with `json.load` and `json.dump`